# Chapter 1: Straightedge and compass

**Source span:** printed pages 1-19, PDF pages 11-29, sections 1.1-1.6. The source pages were used for topic order and coverage only; the prose, diagrams, computations, and checks in this notebook are original.

**Chapter goal:** understand how the straightedge and compass generate Euclidean information from two primitive acts: joining points by straight lines and transferring distances by circles. By the end, the same tools will have produced perpendiculars, parallels, arithmetic on lengths, similar-triangle ratios, and a constructible length that is not rational.

The chapter begins with tools rather than coordinates. A straightedge has no scale, so it can certify collinearity but not length. A compass carries a radius, so it can certify equal distances but not straightness. Most of the chapter is the story of what happens when those two powers cooperate.

## Computational translation guide

| Book object or move | Computational model used here | What we check |
| --- | --- | --- |
| point | a 2-vector such as `np.array([x, y])` | coordinates are finite |
| straightedge line through two points | sampled line segment or line equation | collinearity, dot products, determinants |
| compass circle with center and radius | sampled circle plus distance residuals | every constructed point has the declared radius |
| length transfer | circle intersection with a chosen line | new segment length equals stored radius |
| perpendicular/parallel | dot product zero or determinant zero | angle and direction invariants |
| Thales construction | parallel projection in a triangle | ratios on two rays agree |
| constructible arithmetic | product and quotient as similar-triangle outputs | constructed values match `a*b` and `b/a` |
| irrational diagonal | square diagonal plus parity descent | `d^2 = 2` and no reduced rational representation |

**Library routing.** Matplotlib is the primary renderer because the chapter is planar synthetic Euclidean geometry: the learner needs labeled lines, circles, angles, and proportional segments. SymPy handles exact arithmetic and the square-root identity. NetworkX is used where the irrationality proof is clearer as a dependency graph. Plotly supplies a standalone HTML slider for the applied Thales lab; the PNGs remain the durable validation artifacts.


In [ ]:
from pathlib import Path
import sys, json, math

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Arc, Polygon
import sympy as sp
import networkx as nx
import plotly.graph_objects as go


def find_book_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.name == "The-Four-Pillars-of-Geometry" and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Could not locate course root")


BOOK_ROOT = find_book_root(Path.cwd().resolve())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, image_stats, save_json, save_table
from utils.euclidean import distance, rotate, equilateral_from_segment
from utils.plotting import PALETTE, save_clean

CHAPTER_KEY = "chapter-01-straightedge-and-compass"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_KEY
for child in ["figures", "html", "checks", "tables"]:
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

FIGURES, HTML_ARTIFACTS, TABLES = [], [], []
CHECKS = {}


def fig_ax(width=7.2, height=5.0, title=None):
    fig, ax = plt.subplots(figsize=(width, height), facecolor="#fffdf7")
    ax.set_facecolor("#fffdf7")
    ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.65)
    ax.set_aspect("equal", adjustable="box")
    if title:
        ax.set_title(title, color=PALETTE["ink"], fontsize=13, pad=10, weight="bold")
    return fig, ax


def label_point(ax, name, p, dx=0.05, dy=0.06, color="ink"):
    p = np.asarray(p, dtype=float)
    ax.scatter([p[0]], [p[1]], s=42, color=PALETTE.get(color, color), zorder=5)
    ax.text(p[0] + dx, p[1] + dy, name, color=PALETTE["ink"], fontsize=9.5, weight="bold")


def draw_segment(ax, p, q, color="ink", lw=2.1, ls="-", label=None, shift=(0, 0)):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    ax.plot([p[0], q[0]], [p[1], q[1]], color=PALETTE.get(color, color), lw=lw, ls=ls)
    if label:
        m = (p + q) / 2 + np.asarray(shift, dtype=float)
        ax.text(m[0], m[1], label, color=PALETTE.get(color, color), fontsize=9, weight="bold")


def draw_line_through(ax, p, q, t0=-0.5, t1=1.5, color="gray", lw=1.5, ls="--"):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    v = q - p
    draw_segment(ax, p + t0 * v, p + t1 * v, color=color, lw=lw, ls=ls)


def add_circle(ax, center, radius, color="blue", lw=1.6, ls="-", alpha=1.0, label=None):
    c = np.asarray(center, dtype=float)
    patch = Circle(c, radius, fill=False, ec=PALETTE.get(color, color), lw=lw, ls=ls, alpha=alpha)
    ax.add_patch(patch)
    if label:
        ax.text(c[0] + 0.42 * radius, c[1] + 0.82 * radius, label, color=PALETTE.get(color, color), fontsize=9)


def finish(ax, xlim, ylim):
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel("x"); ax.set_ylabel("y")


def save_figure(fig, filename):
    path = ARTIFACT_ROOT / "figures" / filename
    save_clean(fig, path, dpi=170)
    FIGURES.append(path)
    return path


print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"ARTIFACT_ROOT = {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")


## 1. Straightedge, compass, and length transfer

The first construction assumptions can be read as operations. Given two points, the straightedge joins them and can keep extending the same direction. Given a center and a stored radius, the compass draws a circle. Because the circle is the set of all points at the stored distance, it can move a length without using a marked ruler.

In the figure, the straightedge knows the line through `C` and `D`, while the compass knows the distance `AB`. Placing the compass at `D` marks two points on the line through `C,D`: one has length `CD + AB` from `C`, and the other has length `CD - AB` from `C`. Inspect how the same radius is reused away from its original segment.


In [ ]:
A = np.array([0.0, 1.2]); B = np.array([1.3, 1.2])
C = np.array([0.0, 0.0]); D = np.array([2.2, 0.0])
r = distance(A, B)
v = (D - C) / distance(C, D)
E_plus, E_minus = D + r * v, D - r * v

fig, ax = fig_ax(8.4, 4.6, "Straightedge supplies a line; compass transfers a length")
draw_segment(ax, A, B, color="blue", label="stored radius |AB|", shift=(0, 0.14))
add_circle(ax, D, r, color="orange", label="circle centered at D")
draw_line_through(ax, C, D, t0=-0.12, t1=1.72)
draw_segment(ax, C, D, color="ink", lw=2.4, label="|CD|", shift=(0, -0.18))
draw_segment(ax, D, E_plus, color="orange", lw=2.4, label="+ |AB|", shift=(0, 0.12))
draw_segment(ax, E_minus, D, color="green", lw=2.4, label="- |AB|", shift=(0, -0.20))
for name, point in {"A": A, "B": B, "C": C, "D": D, "CD+AB": E_plus, "CD-AB": E_minus}.items():
    label_point(ax, name, point, dy=0.08 if point[1] >= 1 else 0.05)
ax.annotate("no tick marks on the straightedge", xy=(1.25, -0.28), xytext=(0.3, -0.72),
            arrowprops={"arrowstyle": "->", "lw": 1.5, "color": PALETTE["gray"]},
            color=PALETTE["gray"], fontsize=9)
finish(ax, (-0.45, 4.0), (-0.95, 2.75))
primitive_path = save_figure(fig, "construction-primitives-length-transfer.png")

CHECKS["length_transfer"] = {
    "AB": distance(A, B),
    "CD": distance(C, D),
    "D_to_plus_mark": distance(D, E_plus),
    "D_to_minus_mark": distance(D, E_minus),
    "C_to_plus_mark": distance(C, E_plus),
    "C_to_minus_mark": distance(C, E_minus),
    "plus_expected": distance(C, D) + distance(A, B),
    "minus_expected": distance(C, D) - distance(A, B),
}
CHECKS["length_transfer"]["radius_residual"] = max(
    abs(CHECKS["length_transfer"]["D_to_plus_mark"] - CHECKS["length_transfer"]["AB"]),
    abs(CHECKS["length_transfer"]["D_to_minus_mark"] - CHECKS["length_transfer"]["AB"]),
)
display_artifact(primitive_path, width=760)


## 2. Equilateral triangle from two circles

Euclid's first construction builds an equilateral triangle on a given segment. Draw the circle centered at `A` through `B`, then the circle centered at `B` through `A`. Either intersection can serve as the third vertex. The construction also exposes a hidden geometric assumption: the two circles must actually meet.

The check below records three equal side lengths and two circle residuals for the chosen top intersection. This is the computational version of the proof sentence: `AC` and `BC` are both copies of the same radius `AB`.


In [ ]:
A = np.array([0.0, 0.0]); B = np.array([2.0, 0.0])
C_top = equilateral_from_segment(A, B)
C_bottom = A + rotate(B - A, -math.pi / 3)
r = distance(A, B)

fig, ax = fig_ax(6.8, 5.9, "Equilateral triangle as two circle constraints")
add_circle(ax, A, r, color="blue", label="center A")
add_circle(ax, B, r, color="orange", label="center B")
draw_segment(ax, A, B, color="ink", label="AB")
draw_segment(ax, A, C_top, color="green", label="AC")
draw_segment(ax, B, C_top, color="green", label="BC")
draw_segment(ax, A, C_bottom, color="gray", lw=1.2, ls=":")
draw_segment(ax, B, C_bottom, color="gray", lw=1.2, ls=":")
for name, point in {"A": A, "B": B, "C": C_top, "other intersection": C_bottom}.items():
    label_point(ax, name, point, dy=0.08 if point[1] >= 0 else -0.18)
ax.text(0.52, 1.05, "AC = AB", color=PALETTE["blue"], fontsize=10, weight="bold")
ax.text(1.28, 1.05, "BC = AB", color=PALETTE["orange"], fontsize=10, weight="bold")
finish(ax, (-1.0, 3.0), (-2.0, 2.3))
equilateral_path = save_figure(fig, "equilateral-two-circles.png")

CHECKS["equilateral"] = {
    "AB": distance(A, B),
    "AC": distance(A, C_top),
    "BC": distance(B, C_top),
    "top_circle_A_residual": abs(distance(A, C_top) - r),
    "top_circle_B_residual": abs(distance(B, C_top) - r),
    "area": abs(np.cross(B - A, C_top - A)) / 2,
}
display_artifact(equilateral_path, width=680)


## 3. Bisectors, perpendiculars, and parallels

The same pair of equal-radius circles has a second use. If both intersection points are kept, the line through them is the perpendicular bisector of the original segment. From there we get two routine tools: a perpendicular to a line through a chosen point, and then a parallel by constructing a second perpendicular.

Read the next figure left to right: segment bisector, angle bisector, parallel through a point. The checks use midpoint equality, a dot product for perpendicularity, angle equality, and a determinant for parallel directions.


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13.2, 4.4), facecolor="#fffdf7")
for ax in axs:
    ax.set_facecolor("#fffdf7"); ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.55); ax.set_aspect("equal", adjustable="box")

ax = axs[0]; ax.set_title("segment bisector", fontsize=11, color=PALETTE["ink"], weight="bold")
A = np.array([0.0, 0.0]); B = np.array([2.0, 0.0]); r = distance(A, B)
C = equilateral_from_segment(A, B); D = A + rotate(B - A, -math.pi / 3)
add_circle(ax, A, r, color="blue", alpha=0.7); add_circle(ax, B, r, color="orange", alpha=0.7)
draw_segment(ax, A, B, color="ink"); draw_segment(ax, C, D, color="red", label="perpendicular bisector", shift=(0.08, 0.0))
for name, p in {"A": A, "B": B, "C": C, "D": D}.items():
    label_point(ax, name, p, dy=0.07 if p[1] >= 0 else -0.20)
finish(ax, (-0.75, 2.75), (-1.95, 1.95))

ax = axs[1]; ax.set_title("angle bisector", fontsize=11, color=PALETTE["ink"], weight="bold")
O = np.array([0.0, 0.0]); theta = math.radians(64)
ray1 = np.array([2.3, 0.0]); ray2 = np.array([2.3 * math.cos(theta), 2.3 * math.sin(theta)])
A2 = np.array([1.35, 0.0]); B2 = np.array([1.35 * math.cos(theta), 1.35 * math.sin(theta)])
M2 = (A2 + B2) / 2; bisector_dir = M2 / np.linalg.norm(M2); C2 = 2.25 * bisector_dir
draw_segment(ax, O, ray1, color="ink"); draw_segment(ax, O, ray2, color="ink")
add_circle(ax, O, 1.35, color="gray", ls=":"); draw_segment(ax, A2, B2, color="orange", lw=1.6, ls="--", label="chord AB")
draw_segment(ax, O, C2, color="red", label="equal-angle line", shift=(0.08, 0.05))
for name, p in {"O": O, "A": A2, "B": B2}.items():
    label_point(ax, name, p)
ax.add_patch(Arc(O, 0.72, 0.72, angle=0, theta1=0, theta2=theta/2*180/math.pi, color=PALETTE["green"], lw=2))
ax.add_patch(Arc(O, 0.96, 0.96, angle=0, theta1=theta/2*180/math.pi, theta2=theta*180/math.pi, color=PALETTE["green"], lw=2))
finish(ax, (-0.35, 2.55), (-0.25, 2.45))

ax = axs[2]; ax.set_title("parallel via two perpendiculars", fontsize=11, color=PALETTE["ink"], weight="bold")
L0 = np.array([-0.25, 0.0]); L1 = np.array([2.35, 0.0]); P = np.array([1.15, 1.2])
draw_segment(ax, L0, L1, color="ink", label="L")
draw_segment(ax, np.array([P[0], -0.18]), np.array([P[0], 1.72]), color="blue", label="M perp L", shift=(0.08, 0.0))
draw_segment(ax, np.array([-0.10, P[1]]), np.array([2.55, P[1]]), color="red", label="parallel to L", shift=(0.1, 0.10))
label_point(ax, "P", P); ax.text(1.42, 0.55, "90 deg", fontsize=10, color=PALETTE["blue"], weight="bold")
ax.text(1.42, 1.36, "90 deg", fontsize=10, color=PALETTE["red"], weight="bold")
finish(ax, (-0.45, 2.75), (-0.45, 1.95))

bisector_path = save_figure(fig, "bisectors-perpendiculars-parallels.png")

mid = (A + B) / 2
angle1 = math.atan2(bisector_dir[1], bisector_dir[0])
angle2 = theta - angle1
parallel_det = float(np.linalg.det(np.vstack([L1 - L0, np.array([2.55, P[1]]) - np.array([-0.10, P[1]])])))
CHECKS["bisectors_perpendiculars_parallels"] = {
    "midpoint_to_A": distance(mid, A),
    "midpoint_to_B": distance(mid, B),
    "perpendicular_dot_product": float(np.dot(B - A, C - D)),
    "angle_bisector_radians_left": angle1,
    "angle_bisector_radians_right": angle2,
    "angle_bisector_residual": abs(angle1 - angle2),
    "parallel_direction_determinant": parallel_det,
}
display_artifact(bisector_path, width=960)


## 4. Thales theorem turns parallels into arithmetic

The construction for dividing a segment into equal parts makes a new ray from one endpoint, marks equal compass steps on that ray, joins the last mark to the far endpoint, and draws parallels back to the base. The reason this works is Thales' proportionality theorem: parallel cuts preserve ratios on the sides of a triangle.

The same theorem lets us multiply and divide lengths after a unit segment has been chosen. In the diagrams below, the unit length sits on one ray and the length `a` on another. A parallel transfers the ratio to create `ab` on one ray or `b/a` on the other. This is why constructible geometry already contains a controlled amount of algebra.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.8, 5.0), facecolor="#fffdf7")
for ax in axs:
    ax.set_facecolor("#fffdf7"); ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.55); ax.set_aspect("equal", adjustable="box")

ax = axs[0]; ax.set_title("five equal parts from parallels", fontsize=11, color=PALETTE["ink"], weight="bold")
A = np.array([0.0, 0.0]); B = np.array([5.0, 0.0]); step = np.array([0.55, 0.58])
marks = [A + k * step for k in range(1, 6)]; An = marks[-1]
draw_segment(ax, A, B, color="ink", lw=2.4, label="AB")
draw_segment(ax, A, An, color="blue", lw=2.0, label="equal compass steps", shift=(-0.08, 0.14))
draw_segment(ax, An, B, color="orange", lw=2.0, label="join last mark to B", shift=(0.05, 0.08))
base_points = []
for k, Mk in enumerate(marks[:-1], start=1):
    X = A + (k / 5) * (B - A); base_points.append(X)
    draw_segment(ax, Mk, X, color="gray", lw=1.4, ls="--")
    label_point(ax, f"A{k}", Mk, dx=0.03, dy=0.05, color="blue")
    label_point(ax, f"X{k}", X, dx=-0.03, dy=-0.24, color="green")
label_point(ax, "A", A); label_point(ax, "B", B); label_point(ax, "A5", An, dx=0.03, dy=0.05, color="blue")
for k in range(5):
    x0 = A + k/5 * (B - A); x1 = A + (k+1)/5 * (B - A)
    draw_segment(ax, x0 + np.array([0, -0.12]), x1 + np.array([0, -0.12]), color="green", lw=2.0)
finish(ax, (-0.35, 5.35), (-0.55, 3.45))

ax = axs[1]; ax.set_title("product and quotient by similar triangles", fontsize=11, color=PALETTE["ink"], weight="bold")
O = np.array([0.0, 0.0]); U = np.array([0.0, 1.0]); A3 = np.array([1.6, 0.0])
a = distance(O, A3); b = 1.35
B1 = np.array([0.0, 1.0 + b]); Cprod = np.array([a * b + a, 0.0])
B2 = np.array([a + b, 0.0]); Dquot = np.array([0.0, 1.0 + b / a])
draw_segment(ax, O, np.array([4.25, 0.0]), color="ink", lw=1.8)
draw_segment(ax, O, np.array([0.0, 3.15]), color="ink", lw=1.8)
draw_segment(ax, U, A3, color="blue", lw=2.0, label="reference triangle")
draw_segment(ax, B1, Cprod, color="orange", lw=2.0, label="parallel gives product", shift=(0.2, 0.08))
draw_segment(ax, Dquot, B2, color="green", lw=2.0, label="parallel gives quotient", shift=(0.2, -0.13))
for name, p in {"O": O, "U=1": U, "A=a": A3, "B1=1+b": B1, "C": Cprod, "B2=a+b": B2, "D": Dquot}.items():
    label_point(ax, name, p, dx=0.04, dy=0.05)
draw_segment(ax, A3, Cprod, color="orange", lw=3, label="AC = ab", shift=(0.0, -0.22))
draw_segment(ax, U, Dquot, color="green", lw=3, label="UD = b/a", shift=(0.08, 0.0))
finish(ax, (-0.35, 4.3), (-0.35, 3.4))

thales_path = save_figure(fig, "thales-division-and-arithmetic.png")

base_all = [A, *base_points, B]
intervals = [distance(base_all[i], base_all[i+1]) for i in range(5)]
CHECKS["thales_arithmetic"] = {
    "division_intervals": intervals,
    "division_max_minus_min": max(intervals) - min(intervals),
    "a": a,
    "b": b,
    "constructed_product_ab": distance(A3, Cprod),
    "expected_product_ab": a * b,
    "constructed_quotient_b_over_a": distance(U, Dquot),
    "expected_quotient_b_over_a": b / a,
    "product_residual": abs(distance(A3, Cprod) - a * b),
    "quotient_residual": abs(distance(U, Dquot) - b / a),
}
rows = []
for aa, bb in [(sp.Rational(3, 2), sp.Rational(5, 4)), (sp.Rational(5, 3), sp.Rational(7, 5)), (sp.Rational(4, 1), sp.Rational(3, 2)), (sp.Rational(7, 4), sp.Rational(9, 8)), (sp.Rational(11, 6), sp.Rational(5, 2)), (sp.Rational(13, 10), sp.Rational(21, 8))]:
    rows.append({"a": str(aa), "b": str(bb), "constructible_product": str(aa * bb), "constructible_quotient_b_over_a": str(sp.simplify(bb / aa))})
table_path = save_table(rows, ARTIFACT_ROOT, category="tables", filename="constructible-arithmetic-examples.csv")
TABLES.append(table_path)
display_artifact(thales_path, width=900)


## 5. Similar triangles: same angles, proportional sides

Thales' theorem becomes a theorem about shape. If two triangles have the same angles, one can be moved so that a vertex and two rays coincide. The remaining sides are then parallel, so the side lengths on the two rays must have the same ratio.

The diagram overlays a smaller triangle inside a larger one. The colored side pairs are not measurements taken from the drawing; they are quantities that the parallel-line theorem forces to agree. The exact check uses rational coordinates so the ratio equality is symbolic, not just decimal.


In [ ]:
A = np.array([0.0, 0.0])
B_big = np.array([4.2, 0.0]); C_big = np.array([1.3, 2.7])
scale = sp.Rational(5, 8)
B_small = np.array([float(scale) * B_big[0], float(scale) * B_big[1]])
C_small = np.array([float(scale) * C_big[0], float(scale) * C_big[1]])

fig, ax = fig_ax(7.5, 5.6, "Equal angles force side ratios")
ax.add_patch(Polygon([A, B_big, C_big], closed=True, fill=True, fc=PALETTE["blue"], ec=PALETTE["blue"], alpha=0.11, lw=2))
ax.add_patch(Polygon([A, B_small, C_small], closed=True, fill=True, fc=PALETTE["orange"], ec=PALETTE["orange"], alpha=0.22, lw=2))
draw_segment(ax, B_small, C_small, color="orange", lw=2.5, label="parallel cut", shift=(0.06, 0.08))
draw_segment(ax, B_big, C_big, color="blue", lw=2.5, label="matching side", shift=(0.08, 0.08))
draw_segment(ax, A, B_big, color="ink", lw=2.0); draw_segment(ax, A, C_big, color="ink", lw=2.0)
for name, p in {"A": A, "B": B_big, "C": C_big, "P": B_small, "Q": C_small}.items():
    label_point(ax, name, p, dy=0.07 if name not in {"B", "P"} else -0.20)
ax.text(1.35, -0.33, "AP / AB = 5/8", color=PALETTE["orange"], fontsize=10, weight="bold")
ax.text(0.25, 1.08, "AQ / AC = 5/8", color=PALETTE["orange"], fontsize=10, weight="bold", rotation=63)
finish(ax, (-0.45, 4.7), (-0.55, 3.15))
similar_path = save_figure(fig, "similar-triangles-proportions.png")

AB = sp.Rational(8, 1); AC = sp.Rational(13, 1)
AP = scale * AB; AQ = scale * AC
CHECKS["similar_triangles"] = {
    "AP_over_AB": str(sp.simplify(AP / AB)),
    "AQ_over_AC": str(sp.simplify(AQ / AC)),
    "ratio_identity": bool(sp.simplify(AP / AB - AQ / AC) == 0),
    "area_scale_expected": str(scale ** 2),
    "area_scale_numeric": float(np.cross(B_small - A, C_small - A) / np.cross(B_big - A, C_big - A)),
}
display_artifact(similar_path, width=740)


## 6. The diagonal of the unit square and why it is not rational

A unit square has a constructible diagonal: draw the square, then join opposite vertices. Similar-triangle reasoning gives the equation `d^2 = 2`, so the diagonal length is `sqrt(2)`.

The unsettling part is not that the diagonal exists; it is that no ratio of whole-number lengths equals it. The proof by parity is short enough to turn into a dependency graph. If a reduced fraction `m/n` satisfied `sqrt(2) = m/n`, then `m^2 = 2n^2`, so `m` is even. Writing `m = 2l` forces `n` even too. That contradicts the starting claim that `m` and `n` had no common divisor. The graph shows where the contradiction enters.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.2, 5.2), facecolor="#fffdf7")
for ax in axs:
    ax.set_facecolor("#fffdf7"); ax.grid(True, color="#e8dfcf", lw=0.7, alpha=0.55); ax.set_aspect("equal", adjustable="box")

ax = axs[0]; ax.set_title("constructible diagonal", fontsize=11, color=PALETTE["ink"], weight="bold")
S0 = np.array([0.0, 0.0]); S1 = np.array([1.0, 0.0]); S2 = np.array([1.0, 1.0]); S3 = np.array([0.0, 1.0])
ax.add_patch(Polygon([S0, S1, S2, S3], closed=True, fill=True, fc=PALETTE["green"], ec=PALETTE["green"], alpha=0.13, lw=2))
draw_segment(ax, S0, S2, color="red", lw=2.6, label="d", shift=(0.08, 0.0))
draw_segment(ax, S1, S3, color="gray", lw=1.5, ls=":")
for name, p in {"0": S0, "1": S1, "1+i": S2, "i": S3}.items():
    label_point(ax, name, p, dx=0.03, dy=0.05)
ax.text(0.37, -0.16, "side = 1", color=PALETTE["ink"], fontsize=10, weight="bold")
ax.text(1.06, 0.42, "side = 1", color=PALETTE["ink"], fontsize=10, weight="bold", rotation=90)
ax.text(0.50, 0.56, "d^2 = 1^2 + 1^2", color=PALETTE["red"], fontsize=10, weight="bold", rotation=42)
finish(ax, (-0.25, 1.35), (-0.25, 1.25))

ax = axs[1]; ax.set_title("parity descent proof state", fontsize=11, color=PALETTE["ink"], weight="bold")
G = nx.DiGraph()
labels = {
    "assume": "assume sqrt(2) = m/n\nwith gcd(m,n)=1",
    "square": "m^2 = 2n^2",
    "m_even": "m^2 even\ntherefore m even",
    "substitute": "m = 2l\ntherefore n^2 = 2l^2",
    "n_even": "n^2 even\ntherefore n even",
    "contradiction": "m and n both even\ncontradicts gcd=1",
}
G.add_nodes_from(labels); G.add_edges_from([("assume", "square"), ("square", "m_even"), ("m_even", "substitute"), ("substitute", "n_even"), ("n_even", "contradiction"), ("assume", "contradiction")])
pos = {"assume": (0, 1.6), "square": (1.2, 1.6), "m_even": (2.4, 1.6), "substitute": (1.2, 0.55), "n_even": (2.4, 0.55), "contradiction": (3.75, 1.05)}
colors = [PALETTE["red"] if n == "contradiction" else PALETTE["blue"] for n in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", width=1.5, edge_color=PALETTE["gray"], min_source_margin=8, min_target_margin=8)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2200, node_color=colors, alpha=0.18, edgecolors=colors, linewidths=1.8)
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=8, font_color=PALETTE["ink"])
ax.set_axis_off(); ax.set_xlim(-0.55, 4.45); ax.set_ylim(0.0, 2.05)

sqrt2_path = save_figure(fig, "sqrt2-diagonal-irrationality.png")

CHECKS["sqrt2"] = {
    "diagonal_squared_exact": str(sp.simplify(sp.sqrt(2) ** 2)),
    "sqrt2_is_rational_sympy": bool(sp.sqrt(2).is_rational),
    "square_mod_2_table": {k: (k * k) % 2 for k in [0, 1]},
    "networkx_nodes": G.number_of_nodes(),
    "networkx_edges": G.number_of_edges(),
    "integer_search_denominator_limit": 120,
    "integer_search_exact_solutions": [(mm, nn) for nn in range(1, 121) for mm in range(1, int(math.sqrt(2) * nn) + 2) if mm * mm == 2 * nn * nn],
}
display_artifact(sqrt2_path, width=920)


## 7. Applied construction lab: height from a shadow triangle

A field version of similar triangles is height measurement. Place a unit staff vertically, mark its shadow, and compare it with the shadow of an inaccessible object. The sun rays act like parallel lines, so the small staff triangle and the large object triangle are similar.

The lab below creates a diagram and a Plotly HTML slider. In the static figure, inspect the two right triangles and the shared ray direction. In the HTML artifact, move the sun angle and watch the computed height update while the ratio `height / shadow` stays shared by the two triangles. This is the same Thales idea as the segment-division construction, but now it is being used as an instrument.


In [ ]:
staff_height = 1.0
staff_shadow = 1.45
tower_shadow = 5.8
tower_height = staff_height * tower_shadow / staff_shadow

fig, ax = fig_ax(8.0, 4.9, "Similar-triangle shadow lab")
staff_base = np.array([1.1, 0.0]); staff_top = staff_base + np.array([0.0, staff_height]); staff_tip = staff_base + np.array([staff_shadow, 0.0])
tower_base = np.array([4.2, 0.0]); tower_top = tower_base + np.array([0.0, tower_height]); tower_tip = tower_base + np.array([tower_shadow, 0.0])
draw_segment(ax, np.array([0.0, 0.0]), np.array([10.6, 0.0]), color="ink", lw=2.0, label="ground")
draw_segment(ax, staff_base, staff_top, color="blue", lw=3, label="unit staff")
draw_segment(ax, staff_base, staff_tip, color="blue", lw=2, label="staff shadow", shift=(0.1, -0.18))
draw_segment(ax, staff_top, staff_tip, color="orange", lw=2.2, label="sun ray")
draw_segment(ax, tower_base, tower_top, color="red", lw=3, label="unknown height")
draw_segment(ax, tower_base, tower_tip, color="red", lw=2, label="object shadow", shift=(0.5, -0.18))
draw_segment(ax, tower_top, tower_tip, color="orange", lw=2.2, label="parallel sun ray", shift=(0.4, 0.12))
for name, p in {"staff": staff_base, "1": staff_top, "S": staff_tip, "object": tower_base, "H": tower_top, "T": tower_tip}.items():
    label_point(ax, name, p, dx=0.04, dy=0.05)
ax.text(5.0, 2.6, f"H = 1 * {tower_shadow:.2f} / {staff_shadow:.2f} = {tower_height:.2f}", color=PALETTE["red"], fontsize=11, weight="bold")
finish(ax, (-0.35, 10.55), (-0.35, 4.45))
shadow_path = save_figure(fig, "applied-similar-triangles-shadow-lab.png")

CHECKS["shadow_lab"] = {
    "staff_height": staff_height,
    "staff_shadow": staff_shadow,
    "tower_shadow": tower_shadow,
    "computed_tower_height": tower_height,
    "staff_height_over_shadow": staff_height / staff_shadow,
    "tower_height_over_shadow": tower_height / tower_shadow,
    "ratio_residual": abs((staff_height / staff_shadow) - (tower_height / tower_shadow)),
}

angles = np.linspace(24, 66, 15)
object_height = 4.0
frames = []
for angle in angles:
    slope = math.tan(math.radians(angle))
    staff_shadow_slider = 1.0 / slope
    object_shadow = object_height / slope
    xs = [0, 0, staff_shadow_slider, None, 3, 3, 3 + object_shadow]
    ys = [0, 1.0, 0, None, 0, object_height, 0]
    frames.append(go.Frame(
        data=[go.Scatter(x=xs, y=ys, mode="lines+markers", line={"width": 3})],
        name=f"{angle:.0f}",
        layout=go.Layout(annotations=[dict(x=4.8, y=4.35, showarrow=False, text=f"sun angle {angle:.0f} deg; shared height/shadow ratio = {slope:.3f}")]),
    ))
fig_plotly = go.Figure(data=frames[0].data, frames=frames)
fig_plotly.update_layout(
    title="Shadow measurement as a movable Thales construction",
    xaxis={"range": [-0.5, 12.0], "title": "ground distance", "scaleanchor": "y", "scaleratio": 1},
    yaxis={"range": [-0.3, 5.0], "title": "height"},
    template="plotly_white",
    width=900,
    height=520,
    sliders=[{"active": 0, "currentvalue": {"prefix": "sun angle: "}, "steps": [
        {"label": f"{angle:.0f} deg", "method": "animate", "args": [[f"{angle:.0f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}]}
        for angle in angles
    ]}],
)
fig_plotly.add_annotation(x=0.15, y=1.18, text="unit staff", showarrow=False)
fig_plotly.add_annotation(x=3.25, y=4.22, text="unknown object", showarrow=False)
html_path = ARTIFACT_ROOT / "html" / "shadow-thales-slider.html"
fig_plotly.write_html(html_path, include_plotlyjs="cdn", full_html=True)
HTML_ARTIFACTS.append(html_path)
lab_json_path = save_json(CHECKS["shadow_lab"], ARTIFACT_ROOT, category="checks", filename="shadow-lab-invariants.json")

display_artifact(shadow_path, width=760)
display_artifact(html_path, width="100%", height=520)


## Construction checks and artifact manifest

The notebook has been using diagrams as mathematical instruments. This final cell makes the promise explicit: every artifact exists, every PNG is nonblank at a usable size, and the main geometric claims have executable checks. The assertions are intentionally plain so that a reader can connect a failing check to a visible construction.


In [ ]:
checks_path = save_json(CHECKS, ARTIFACT_ROOT, category="checks", filename="construction-invariants.json")
manifest = {
    "chapter_key": CHAPTER_KEY,
    "figures": [str(path.relative_to(ARTIFACT_ROOT)) for path in FIGURES],
    "html": [str(path.relative_to(ARTIFACT_ROOT)) for path in HTML_ARTIFACTS],
    "checks": ["checks/construction-invariants.json", "checks/shadow-lab-invariants.json"],
    "tables": [str(path.relative_to(ARTIFACT_ROOT)) for path in TABLES],
}
manifest_path = save_json(manifest, ARTIFACT_ROOT, category="checks", filename="artifact_manifest.json")
all_artifacts = FIGURES + HTML_ARTIFACTS + TABLES + [checks_path, manifest_path, ARTIFACT_ROOT / "checks" / "shadow-lab-invariants.json"]
assert_artifacts(all_artifacts, min_size=128)

stats = [image_stats(path) for path in FIGURES]
assert len(FIGURES) >= 6
assert len(HTML_ARTIFACTS) >= 1
assert all(item["width"] >= 300 and item["height"] >= 240 for item in stats)
assert all(item["pixel_std"] > 2.0 for item in stats)

lt = CHECKS["length_transfer"]
assert lt["radius_residual"] < 1e-12
assert abs(lt["C_to_plus_mark"] - lt["plus_expected"]) < 1e-12
assert abs(lt["C_to_minus_mark"] - lt["minus_expected"]) < 1e-12

eq = CHECKS["equilateral"]
assert max(abs(eq["AB"] - eq["AC"]), abs(eq["AB"] - eq["BC"])) < 1e-12
assert eq["area"] > 0

bp = CHECKS["bisectors_perpendiculars_parallels"]
assert abs(bp["midpoint_to_A"] - bp["midpoint_to_B"]) < 1e-12
assert abs(bp["perpendicular_dot_product"]) < 1e-12
assert bp["angle_bisector_residual"] < 1e-12
assert abs(bp["parallel_direction_determinant"]) < 1e-12

th = CHECKS["thales_arithmetic"]
assert th["division_max_minus_min"] < 1e-12
assert th["product_residual"] < 1e-12
assert th["quotient_residual"] < 1e-12

sim = CHECKS["similar_triangles"]
assert sim["ratio_identity"] is True
assert abs(sim["area_scale_numeric"] - float(sp.Rational(25, 64))) < 1e-12

sq = CHECKS["sqrt2"]
assert sq["diagonal_squared_exact"] == "2"
assert sq["sqrt2_is_rational_sympy"] is False
assert sq["integer_search_exact_solutions"] == []
assert sq["networkx_nodes"] >= 6 and sq["networkx_edges"] >= 5

sh = CHECKS["shadow_lab"]
assert sh["ratio_residual"] < 1e-12

print(json.dumps({
    "figures": len(FIGURES),
    "html": len(HTML_ARTIFACTS),
    "checks": [str(checks_path.relative_to(BOOK_ROOT)), str(manifest_path.relative_to(BOOK_ROOT))],
    "tables": [str(path.relative_to(BOOK_ROOT)) for path in TABLES],
}, indent=2))


## Takeaways

- A straightedge records incidence and direction; a compass records distance. Their separation is the starting point of Euclidean construction.
- The equilateral triangle construction works because two circle constraints create a point whose distances to both endpoints equal the original side.
- Perpendiculars, angle bisectors, and parallels are not new primitives here; they are built from circle intersections and repeated perpendicular construction.
- Thales' theorem is the bridge from construction to arithmetic: parallel cuts preserve ratios, enabling equal division, multiplication, and quotient constructions after a unit segment is chosen.
- Similar triangles generalize the same ratio idea from one triangle cut by a parallel to any pair of triangles with matching angles.
- The unit-square diagonal is constructible and satisfies `d^2 = 2`, but the parity proof shows that this constructible length is not rational.
- The applied shadow lab is a field version of the same geometry: parallel rays make a small triangle into a measuring device for a larger one.
